In [1]:
!pip install -q \
  torch \
  peft \
  huggingface_hub \
  ipywidgets

!pip install -U datasets optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0

In [2]:
import os, shutil, gc, torch, optuna
from huggingface_hub import login, notebook_login, HfFolder, HfApi, hf_hub_download, delete_repo, list_repo_files, snapshot_download
os.environ["TRANSFORMERS_NO_TF"] = "1"   # prevents TF/Keras import
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainerCallback,
                          Trainer, TrainingArguments, DataCollatorWithPadding, default_data_collator)
from peft import (LoraConfig, get_peft_model, prepare_model_for_kbit_training)
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
from google.colab import files
import json
import time
import re
import inspect
import pandas as pd
import numpy as np
import tempfile
from datetime import datetime
from collections import Counter
from torch.utils.data import Dataset, Subset

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [3]:
api = HfApi()
login()

In [ ]:
HF_DATASET_REPO = "eduhuemar001/news"

# Download JSON files from HF dataset repo
train_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="train.json",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False
)
eval_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="eval.json",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False
)

# Load datasets
with open(train_path, "r", encoding="utf-8") as f:
    dataset_train = json.load(f)

with open(eval_path, "r", encoding="utf-8") as f:
    dataset_eval = json.load(f)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


train.json:   0%|          | 0.00/70.9M [00:00<?, ?B/s]

eval.json:   0%|          | 0.00/23.1M [00:00<?, ?B/s]

In [ ]:
class NewsPairDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data                  # list of dicts with keys of title, text, status
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        item = self.data[i]
        title = (item.get("title") or "").strip()
        text  = (item.get("text")  or "").strip()
        label = int(item.get("status"))   # 0/1

        enc = self.tok(
            text=title,                   # News title
            text_pair=text,               # News article
            truncation="only_second",     # keep full title, truncate only news article
            max_length=self.max_length,
            padding=False,                # let collator pad
            return_attention_mask=True
        )

        out = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long),
        }
        return out

train_dataset = NewsPairDataset(dataset_train, tokenizer, max_length=512)
eval_dataset  = NewsPairDataset(dataset_eval,  tokenizer, max_length=512)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model = model.to("cuda")

best = {
    "learning_rate": 0.0002111577160148881,
    "weight_decay": 0.14170474728616472,
    "warmup_ratio": 0.15983285501618702,
    "lr_scheduler_type": "cosine",
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 6
}

class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs:
            print(f"[step {state.global_step}] train_loss={logs['loss']:.4f}")
        if "eval_loss" in logs:
            ep = logs.get("epoch", state.epoch)
            print(f"[epoch {ep:.2f}] eval_loss={logs['eval_loss']:.4f}")

args = TrainingArguments(
    output_dir="./distilbert_final",
    learning_rate=best["learning_rate"],
    weight_decay=best["weight_decay"],
    warmup_ratio=best["warmup_ratio"],
    lr_scheduler_type=best["lr_scheduler_type"],
    per_device_train_batch_size=best["per_device_train_batch_size"],
    per_device_eval_batch_size=best["per_device_eval_batch_size"],
    gradient_accumulation_steps=best["gradient_accumulation_steps"],
    num_train_epochs=best["num_train_epochs"],
    eval_strategy="epoch",
    logging_steps=20,
    logging_first_step=True,
    report_to="none",
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    seed=42,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    callbacks=[PrintLossCallback()],
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2450808969.py:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

HF_REPO_ID = "eduhuemar001/distilbert-news"
LOCAL_PATH = "distilbert-news-local"
resume_checkpoint = None

# Train
start_time = time.time()
print("Starting training...")
trainer.train()
end_time = time.time()
print(f"Training took {end_time - start_time:.2f} seconds")

# Save LoRA adapter (not full model)
model.save_pretrained(LOCAL_PATH)
tokenizer.save_pretrained(LOCAL_PATH)

No checkpoint found.
Starting fresh training...


Epoch,Training Loss,Validation Loss
1,0.045500,0.028390
2,0.011700,0.012122
3,0.026800,0.017217
4,0.019500,0.015818
5,0.002200,0.012040
6,0.019100,0.013071


[step 1] train_loss=0.6759
[step 20] train_loss=0.6838
[step 40] train_loss=0.6119
[step 60] train_loss=0.2957
[step 80] train_loss=0.1126
[step 100] train_loss=0.0292
[step 120] train_loss=0.0204
[step 140] train_loss=0.0174
[step 160] train_loss=0.0064
[step 180] train_loss=0.0101
[step 200] train_loss=0.0053
[step 220] train_loss=0.0010
[step 240] train_loss=0.0007
[step 260] train_loss=0.0102
[step 280] train_loss=0.0011
[step 300] train_loss=0.0003
[step 320] train_loss=0.0078
[step 340] train_loss=0.0118
[step 360] train_loss=0.0081
[step 380] train_loss=0.0145
[step 400] train_loss=0.0170
[step 420] train_loss=0.0067
[step 440] train_loss=0.0158
[step 460] train_loss=0.0011
[step 480] train_loss=0.0018
[step 500] train_loss=0.0155
[step 520] train_loss=0.0230
[step 540] train_loss=0.0118
[step 560] train_loss=0.0321
[step 580] train_loss=0.0132
[step 600] train_loss=0.0313
[step 620] train_loss=0.0218
[step 640] train_loss=0.0013
[step 660] train_loss=0.0543
[step 680] train_los

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s-local/model.safetensors:   0%|          |  575kB /  268MB            

Deleting checkpoint repo: eduhuemar001/distilbert-news-checkpoint


In [ ]:
LOCAL_DIR = "distilbert_final"
api.upload_folder(
    folder_path=LOCAL_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model",
    path_in_repo="checkpoints",
    commit_message="Upload DistilBERT training folder with all checkpoints",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nt-5052/model.safetensors:   3%|3         | 8.33MB /  268MB            

  ...kpoint-4210/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...kpoint-1684/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

...

  ...nt-4210/model.safetensors:   0%|          |  574kB /  268MB            

  ...nt-1684/model.safetensors:   0%|          |  574kB /  268MB            

  ...ckpoint-5052/optimizer.pt:   0%|          |  571kB /  536MB            

  ...eckpoint-842/optimizer.pt:   0%|          |  558kB /  536MB            

CommitInfo(commit_url='https://huggingface.co/eduhuemar001/distilbert-news/commit/a2bf31480b03e989aa35683900cb87ef3a44c5b3', commit_message='Upload DistilBERT training folder with all checkpoints', commit_description='', oid='a2bf31480b03e989aa35683900cb87ef3a44c5b3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/eduhuemar001/distilbert-news', endpoint='https://huggingface.co', repo_type='model', repo_id='eduhuemar001/distilbert-news'), pr_revision=None, pr_num=None)

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

with open(file_name, "r", encoding="utf-8") as f:
    log_text = f.read()

train_matches = re.findall(r"\[step\s+(\d+)\]\s*train_loss=([\d.]+)", log_text)
train_data = [{"step": int(s), "train_loss": float(l)} for s, l in train_matches]

pd.DataFrame(train_data).to_csv("train_losses.csv", index=False)
files.download("train_losses.csv")

Saving train_loss_raw.txt to train_loss_raw (1).txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
api.upload_file(
    path_or_fileobj="train_losses.csv",
    path_in_repo="logs/train_losses.csv",
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Add training loss log CSV"
)

CommitInfo(commit_url='https://huggingface.co/eduhuemar001/distilbert-news/commit/f58cfb5ed5219cc6f0cb3e514bfc3dccb14534e2', commit_message='Add training loss log CSV', commit_description='', oid='f58cfb5ed5219cc6f0cb3e514bfc3dccb14534e2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/eduhuemar001/distilbert-news', endpoint='https://huggingface.co', repo_type='model', repo_id='eduhuemar001/distilbert-news'), pr_revision=None, pr_num=None)

In [5]:
local_file = "best_params.json"
with open(local_file, "w", encoding="utf-8") as f:
    json.dump(best, f, ensure_ascii=False, indent=2)

api.upload_file(
    path_or_fileobj=local_file,
    path_in_repo="tuning/best_params.json",
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Add best HPO params"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


CommitInfo(commit_url='https://huggingface.co/eduhuemar001/distilbert-news/commit/b77f5cc31d451ea8e43cac84ff500fb66ddffafe', commit_message='Add best HPO params', commit_description='', oid='b77f5cc31d451ea8e43cac84ff500fb66ddffafe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/eduhuemar001/distilbert-news', endpoint='https://huggingface.co', repo_type='model', repo_id='eduhuemar001/distilbert-news'), pr_revision=None, pr_num=None)